# DS2002 · Pandas Challenge

**Lab — 2026-09-18 · Fall 2026**  

---

## Lab 04 — Pandas Challenge

Four hundred generated orders. Each question builds toward a demand report you could hand a vendor.

The data is seeded, so everyone's numbers should match. That is deliberate: if your total revenue differs from your neighbor's, one of you has a bug, and the assertions at the end will tell you which.

Every answer needs the number **and** a sentence saying what it means. A cell that prints `4218.5` with no interpretation is half an answer.

In [ ]:
import pandas as pd, numpy as np
rng = np.random.default_rng(4)
n = 400
df = pd.DataFrame({
    'vendor_id': rng.choice(['V-01','V-05','V-10','V-18'], n),
    'category': rng.choice(['Food','Merch','RainGear','Drink'], n, p=[.5,.2,.1,.2]),
    'qty': rng.integers(1, 4, n),
    'price': rng.choice([4.5, 6.0, 7.5, 12.0, 24.0], n),
})
df.head()

,vendor_id,category,qty,price
0,V-10,Drink,2,24.0
1,V-18,RainGear,1,12.0
2,V-18,Drink,3,4.5
3,V-10,Food,2,12.0
4,V-18,Drink,3,7.5


### Q1 — Add `revenue`, then report total revenue and total units.

*Expected: 400 rows, and revenue should land between $8,000 and $9,000.*

In [ ]:
df['revenue'] = df['qty'] * df['price']
total_revenue = df['revenue'].sum()
total_units = df['qty'].sum()

print(f"Total revenue: ${total_revenue:,.2f}")
print(f"Total units: {total_units}")
print(f"Number of rows: {len(df)}")
print(f"The total revenue of ${total_revenue:,.2f} and {total_units} units come from {len(df)} orders.")

Total revenue: $8,520.00
Total units: 783
Number of rows: 400
The total revenue of $8,520.00 and 783 units come from 400 orders.


What it means: My code above is calculating and printing the result of 400 orders, and there being 783 units sold, which resulted in a revenue of $8520

### Q2 — Revenue by category, highest to lowest.

Include the share of total as a percentage in the same table.

In [ ]:
revenue_by_category = df.groupby('category')['revenue'].sum().sort_values(ascending=False).reset_index()
revenue_by_category['share_of_total'] = (revenue_by_category['revenue'] / total_revenue * 100).round(2)

display(revenue_by_category)


,category,revenue,share_of_total
0,Food,4293.0,50.39
1,Merch,1771.5,20.79
2,Drink,1554.0,18.24
3,RainGear,901.5,10.58


What it means: My code above creates a table that shows the total revenue that is made from each product category. It is ordered from the highest revenue to the lowest and shows the percentage of total revenue that each category is responsible for making.

### Q3 — Which vendor has the highest *average* order revenue?

Report the average alongside the order count for each vendor. A high average on twelve orders is a different claim from a high average on two hundred.

In [ ]:
vendor_revenue_stats = df.groupby('vendor_id').agg(
    total_revenue=('revenue', 'sum'),
    order_count=('vendor_id', 'count')
).reset_index()
vendor_revenue_stats['average_order_revenue'] = vendor_revenue_stats['total_revenue'] / vendor_revenue_stats['order_count']

sorted_vendor_stats = vendor_revenue_stats.sort_values(by='average_order_revenue', ascending=False)

display(sorted_vendor_stats)


,vendor_id,total_revenue,order_count,average_order_revenue
0,V-01,2124.0,94,22.595745
3,V-18,2349.0,108,21.750000
1,V-05,1914.0,93,20.580645
2,V-10,2133.0,105,20.314286


What it means: My code above shows the total revenue, number of orders, and the average revenue per order for each of the vendors. They are organized from the highest average to the lowest average for revenue.

### Q4 — What share of revenue comes from Merch?

Print it as a percentage rounded to one decimal.

In [ ]:
merch_share = revenue_by_category[revenue_by_category['category'] == 'Merch']['share_of_total'].iloc[0]
print(f"Revenue from Merch: {merch_share:.1f}%")


Revenue from Merch: 20.8%


What it means: My code above shows that the "merch" category makes up 20.8% of the total revenue that is made.

### Q5 — Join in the vendor names.

The frame only has `vendor_id`. Merge the lookup below so your report is readable.

**Requirements:** left join, `validate='many_to_one'`, and prove the row count and revenue total did not change. One vendor id in the orders is not in this lookup — find it, and decide what to do about it.

In [ ]:
vendor_names = pd.DataFrame({
    'vendor_id': ['V-01', 'V-05', 'V-10'],
    'vendor_name': ['Hoos Burgers', 'Rotunda Tacos', 'Cav Merch North'],
})

merged_df = pd.merge(df, vendor_names, on='vendor_id', how='left', validate='many_to_one')

# Validate row count and revenue total
print(f"Original df row count: {len(df)}")
print(f"Merged df row count: {len(merged_df)}")
print(f"Original df total revenue: {df['revenue'].sum():.2f}")
print(f"Merged df total revenue: {merged_df['revenue'].sum():.2f}")

# Find unmatched vendor_id
unmatched_vendor_ids = df[~df['vendor_id'].isin(vendor_names['vendor_id'])]['vendor_id'].unique()
print(f"Unmatched vendor IDs: {unmatched_vendor_ids}")

display(merged_df.head())


Original df row count: 400
Merged df row count: 400
Original df total revenue: 8520.00
Merged df total revenue: 8520.00
Unmatched vendor IDs: ['V-18']


,vendor_id,category,qty,price,revenue,vendor_name
0,V-10,Drink,2,24.0,48.0,Cav Merch North
1,V-18,RainGear,1,12.0,12.0,NaN
2,V-18,Drink,3,4.5,13.5,NaN
3,V-10,Food,2,12.0,24.0,Cav Merch North
4,V-18,Drink,3,7.5,22.5,NaN


What it means: My code above combines the vendor names with the confirmed data consistency by using the row counts, total revenue, and the identified "V-18" as the unmatched vendor, which is kept in the dataset.

### Q6 — A pivot table: vendors down the side, categories across the top, revenue in the cells.

Add row and column totals so it reads as a report rather than a grid of numbers.

In [ ]:
revenue_pivot_table = pd.pivot_table(
    merged_df,
    index='vendor_name',
    columns='category',
    values='revenue',
    aggfunc='sum',
    margins=True,  # Add row and column totals
    fill_value=0 # Fill NaN values with 0 for better readability
)

display(revenue_pivot_table)

category,Drink,Food,Merch,RainGear,All
vendor_name,,,,,
Cav Merch North,502.5,1054.5,400.5,175.5,2133.0
Hoos Burgers,171.0,1338.0,373.5,241.5,2124.0
Rotunda Tacos,298.5,882.0,489.0,244.5,1914.0
All,972.0,3274.5,1263.0,661.5,6171.0


What it means: My code is creating a pivot table that is responsible for organizing revenue based on the vendor and category. It shows the sum for each intersection and the overall row and column totals for a completed list.




### Q7 — Validate your work

**TODO:** uncomment and make these pass. Assign your results to the named variables as you go.

In [ ]:
assert len(df) == 400
assert 8000 < df['revenue'].sum() < 9000, df['revenue'].sum()
assert abs(revenue_by_category['revenue'].sum() - df['revenue'].sum()) < 0.01
assert len(merged_df) == len(df), 'the vendor merge changed the row count'
print('checks passed.')

checks passed.


### Write-up

**a)** What would you tell these vendors to do differently next game? One paragraph, with at least two numbers from your report in it.

**b)** Which of your seven answers is the least trustworthy, and why? Point at a specific weakness — a small group size, an unmatched vendor, a category that is really two things.

a) I think it would be helpful for vendors to analyze their sales by category and average order revenue to help understand trends for future game-day performance. Specifically, looking at HoosBurgers which had the highest average order revenue, with $22.60 per order. This was due to their Food sales. It could be good for other vendors to focus on food, since that is clearly bringing in the most revenue, since it makes up 50.39% of the sales. It could be helpful for other vendors to analyze how other categories like food specifically are doing to see where there is demand.

b)I think the least trustworthy answer is the pivot table from the sixth question, because the first vendor lookup from question 5 showed there was a none-matching vendor ID, V-18. And the pivot table uses vendor_name as its index, all of the order from V-18 are then not in the list of named vendor rows and are only slightly included in the 'All' total. Since there is a decent portion of the data missing, it means that individual vendor breakdowns that are done by name in the pivot table are not complete and don't show the full revenue across all the vendors, so the findings from this table could be potentially misguiding.